# Meteostat

In [35]:
import pandas as pd
from geopy.geocoders import Nominatim
import folium
from meteostat import Stations, Daily, Point
from datetime import datetime
import time
import re
from unidecode import unidecode
import matplotlib.pyplot as plt

In [36]:
# Initialize geocoder
geolocator = Nominatim(user_agent="weather_locator")

## Clean Cities

In [37]:
# Load your city data
df = pd.read_excel("meteo_colombia/cities.xlsx")  # Must have 'City', 'Departamento', 'Country'

In [38]:
df

,Country,Departamento,City
0,COLOMBIA,BOGOTA,BOGOTA
1,COLOMBIA,CUNDINAMARCA,MOSQUERA
2,COLOMBIA,CUNDINAMARCA,SOACHA
3,COLOMBIA,CUNDINAMARCA,CHIA
4,COLOMBIA,CUNDINAMARCA,* CUNDINAMARCA. MUNICIPIO DESCONOCIDO
...,...,...,...
1178,DESCONOCIDO,EXTERIOR,* EXTERIOR. PAÍS DESCONOCIDO
1179,COLOMBIA,SANTANDER,ENCINO
1180,DINAMARCA,EXTERIOR,EXTERIOR_DINAMARCA
1181,ITALIA,EXTERIOR,EXTERIOR_ITALIA


In [39]:
df['id_city'] = df['Country'] + '_' + df['Departamento'] + '_' + df['City']

In [40]:
# Function to clean text
def clean_text(text):
    text = str(text).lower()               # Lowercase
    text = unidecode(text)                 # Remove accents
    text = re.sub(r'[^\w\s]', '', text)    # Remove punctuation
    text = re.sub(r'\s+', ' ', text)       # Normalize whitespace
    return text.strip()

# Apply to relevant columns
df['City'] = df['City'].apply(clean_text)
df['Departamento'] = df['Departamento'].apply(clean_text)
df['Country'] = df['Country'].apply(clean_text)

# Optional: remove duplicate cities (if any)
df.drop_duplicates(subset=['City', 'Departamento', 'Country'], inplace=True)

# Geocode

In [41]:
# Function to geocode city
def geocode_city(city, depto, country):
    query = f"{city}, {depto}, {country}"
    try:
        location = geolocator.geocode(query)
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        print(f"Geocoding error for {query}: {e}")
    return None, None

In [42]:
cities_df = df.copy()
cities_df = cities_df.iloc[:10]  # Limit to first 10 for testing

In [43]:
cities_df

,Country,Departamento,City,id_city
0,colombia,bogota,bogota,COLOMBIA_BOGOTA_BOGOTA
1,colombia,cundinamarca,mosquera,COLOMBIA_CUNDINAMARCA_MOSQUERA
2,colombia,cundinamarca,soacha,COLOMBIA_CUNDINAMARCA_SOACHA
3,colombia,cundinamarca,chia,COLOMBIA_CUNDINAMARCA_CHIA
4,colombia,cundinamarca,cundinamarca municipio desconocido,COLOMBIA_CUNDINAMARCA_* CUNDINAMARCA. MUNICIPI...
5,colombia,tolima,purificacion,COLOMBIA_TOLIMA_PURIFICACION
6,colombia,cundinamarca,funza,COLOMBIA_CUNDINAMARCA_FUNZA
7,colombia,arauca,arauca,COLOMBIA_ARAUCA_ARAUCA
8,colombia,cundinamarca,sopo,COLOMBIA_CUNDINAMARCA_SOPO
9,colombia,cundinamarca,la calera,COLOMBIA_CUNDINAMARCA_LA CALERA


In [48]:
# Create empty columns for coordinates
cities_df['Latitude'] = None
cities_df['Longitude'] = None

# Geocode each city and store coordinates
for idx, (city, depto, country) in cities_df[['City', 'Departamento', 'Country']].iterrows():
    lat, lon = geocode_city(city, depto, country)
    cities_df.at[idx, 'Latitude'] = lat
    cities_df.at[idx, 'Longitude'] = lon
    time.sleep(1)  # To respect Nominatim's usage policy

# Create a map centered at the mean location
mean_lat = cities_df['Latitude'].dropna().mean()
mean_lon = cities_df['Longitude'].dropna().mean()
city_map = folium.Map(location=[mean_lat, mean_lon], zoom_start=6)

# Add markers for each city
for _, row in cities_df.iterrows():
    if pd.notnull(row['Latitude']) and pd.notnull(row['Longitude']):
        folium.Marker(
            [row['Latitude'], row['Longitude']],
            popup=f"{row['City']}, {row['Departamento']}, {row['Country']}"
        ).add_to(city_map)

# Save and display the map
city_map.save('cities_map.html')

# In a Jupyter Notebook, directly display:
city_map

# Meteo

In [49]:
cities_df

,Country,Departamento,City,id_city,Latitude,Longitude
0,colombia,bogota,bogota,COLOMBIA_BOGOTA_BOGOTA,4.653382,-74.083633
1,colombia,cundinamarca,mosquera,COLOMBIA_CUNDINAMARCA_MOSQUERA,4.7002,-74.238506
2,colombia,cundinamarca,soacha,COLOMBIA_CUNDINAMARCA_SOACHA,4.582141,-74.219718
3,colombia,cundinamarca,chia,COLOMBIA_CUNDINAMARCA_CHIA,4.866033,-74.030628
4,colombia,cundinamarca,cundinamarca municipio desconocido,COLOMBIA_CUNDINAMARCA_* CUNDINAMARCA. MUNICIPI...,None,None
5,colombia,tolima,purificacion,COLOMBIA_TOLIMA_PURIFICACION,3.858211,-74.930822
6,colombia,cundinamarca,funza,COLOMBIA_CUNDINAMARCA_FUNZA,4.717747,-74.203155
7,colombia,arauca,arauca,COLOMBIA_ARAUCA_ARAUCA,7.085199,-70.757776
8,colombia,cundinamarca,sopo,COLOMBIA_CUNDINAMARCA_SOPO,4.893144,-73.953414
9,colombia,cundinamarca,la calera,COLOMBIA_CUNDINAMARCA_LA CALERA,4.706824,-73.932085


In [50]:
# Define the date range
start = datetime(2023, 1, 1)
end = datetime(2023, 12, 31)

In [51]:
# create dir if not exist 'meteo_colombia/cities/'
import os
if not os.path.exists('meteo_colombia/cities/'):
    os.makedirs('meteo_colombia/cities/')


In [53]:
for i in range(len(cities_df)):
    
    lat = cities_df['Latitude'][i]
    lon = cities_df['Longitude'][i]
    # if lat or lon is None, skip the city
    if lat is None or lon is None:
        print(f"Skipping {cities_df['City'][i]} due to missing coordinates.")
        continue
    # create point for city
    city = Point(lat, lon)

    # Get daily data for
    data = Daily(city, start, end)
    data = data.fetch()
    # save data to csv in 'meteo_colombia/cities/' folder
    data.to_csv(f'meteo_colombia/cities/{cities_df["id_city"][i]}.csv')
    

Skipping cundinamarca municipio desconocido due to missing coordinates.
